In [270]:
import re
import pandas as pd
import numpy as np

In [271]:
years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

driver_code_map = {
    1: "permanent_agriculture",
    2: "hard_commodities",
    3: "shifting_cultivation",
    4: "logging",
    5: "wildfire",
    6: "settlements_infrastructure",
    7: "other_natural_disturbances"
}

sdpt_type_code_map = {
    1: "planted_forest",
    2: "tree_crop"
}

sdpt_name_code_map = {
    1: "oil_palm",
    2: "wood_fiber",
    3: "other"
}

In [ ]:
""" Regex-based land-use reclassification rules
These rules replace the default land use classes.
1. Convert annual GLAD LC values to LU tokens.
2. Use regex to identify token patterns for exceptions.
3. Reclassify token arrays and assign a matching node_code array.

Node codes used here:
1) Settlements and Infrastructure:
    10 = Built from GLAD data

2) Cropland:
    20  = Crop from GLAD data
    21  = Crop from oil palm extent
    22  = Crop from SDPT tree crop extent
    23  = Crop from pre-2000 plantation
    24X = Crop from permanent agriculture driver
         - 243 = TV after TCL where driver is permanent agriculture (assume tree crops)
         - 244 = SV after TCL outside GPW extent (i.e. "rangeland") where driver is permanent agriculture (assume crops)


3) Forest:
    30  = Tall veg from GLAD data
    31  = Forest from SDPT planted forest extent
    32  = Forest from GMW mangrove extent
    33X = Short vegetation or bare reclassified as Forest using drivers rules (assume unstocked forest)
        333 = Forest from shifting cultivation driver
        334 = Forest from logging driver
        335 = Forest from wildfire driver
        337 = Forest from natural disturbance driver

4) Grassland:
    40 = Short veg from GLAD data

5) Wetland:
    50 = Wetland from GLAD data

6) Other
    60 = Bare from GLAD data
    61 = Water from GLAD data
    62 = Snow/ice from GLAD data
"""

# Settlements > Cropland > Forest Land > Grassland > Wetlands > Other
# Default GLAD LC numeric values
settlement_lc   = {250}                                         # Built up
cropland_lc     = {244}                                         # Cropland
forest_lc       = set(range(27, 49)) | set(range(127, 149))     # Tall vegetation
grass_lc        = set(range(5, 27)) | set(range(105, 127))      # Short veg
wetland_lc      = set(range(200, 205))                          # Wetland
bare_lc         = set(range(0, 5)) | set(range(100, 105))       # Bare
water_lc        = set(range(205, 208))                          # Open water
ice_lc          = {241}                                         # Snow/ice

# Lookup table to go from GLAD LC code -> default LU token
lc_token_map = {
    **{v: "S" for v in settlement_lc},
    **{v: "C" for v in cropland_lc},
    **{v: "F" for v in forest_lc},
    **{v: "G" for v in grass_lc},
    **{v: "W" for v in wetland_lc},
    **{v: "B" for v in bare_lc},
    **{v: "O" for v in water_lc},
    **{v: "I" for v in ice_lc},
}

node_code_map = {
    "built_glad": 10,

    "crop_glad": 20,
    "crop_oil_palm": 21,
    "crop_sdpt_tree_crop": 22,
    "crop_pre_2000_plantation": 23,
    "crop_perm_ag_driver": 24,

    "forest_glad": 30,
    "forest_gmw_mangrove": 31,
    "forest_sdpt_planted_forest": 32,
    "forest_shift_cult_driver": 333,
    "forest_logging_driver": 334,
    "forest_wildfire_driver": 335,
    "forest_nat_dist_driver": 337,

    "grass_glad": 40,

    "wetland_glad": 50,

    "bare_glad": 60,
    "water_glad": 61,
    "ice_glad": 62,
}

# Default node codes before rules are applied
def default_node_code(token):
    if token == "S":
        return node_code_map["built_glad"]
    if token == "C":
        return node_code_map["crop_glad"]
    if token == "F":
        return node_code_map["forest_glad"]
    if token == "G":
        return node_code_map["grass_glad"]
    if token == "W":
        return node_code_map["wetland_glad"]
    if token == "B":
        return node_code_map["bare_glad"]
    if token == "O":
        return node_code_map["water_glad"]
    if token == "I":
        return node_code_map["ice_glad"]
    return None

# Function to get land use token per land cover numeric value (tokens used for regex exception rules)
def token_for_lc(v):
    return lc_token_map.get(v, "-")

def set_tokens(tokens, node_codes, indices, new_token, node_code):
    for i in indices:
        tokens[i] = new_token
        node_codes[i] = node_code

In [273]:
# Helpers
def as_bool(v):
    if pd.isna(v):
        return False
    if isinstance(v, str):
        return v.strip().lower() in {"true", "t", "yes", "y", "1"}
    return bool(v)
# TODO: Rewrite or delete after switching from table data to geospatial data

def as_int_or_none(v):
    if pd.isna(v):
        return None
    try:
        return int(v)
    except (TypeError, ValueError):
        return None
# TODO: Rewrite or delete after switching from table data to geospatial data

# Check if there was tree cover loss by the start of the timeseries.
def tcl_prior_to_timeseries(tcl_year, start_year=2015):
    tcl_year = int(tcl_year)
    return tcl_year <= start_year
#TODO: May want to consider changing start year to 2016 (i.e. +1)

In [ ]:
def apply_extent_rules(ctx):
    tokens = ctx["tokens"]
    node_codes = ctx["node_codes"]

    crop_reclass_idx = [i for i, token in enumerate(tokens) if token in {"F", "G", "W", "B"}]
    forest_reclass_idx = [i for i, token in enumerate(tokens) if token in {"G", "W", "B"}]
    # TODO: May want to consider not including wetland?

    # Crop is highest priority and extents are applied in this order: oil palm -> SDPT tree crop --> pre-2000 plantation
    if ctx["oil_palm"]:
        set_tokens(tokens, node_codes, crop_reclass_idx, "C", node_code_map["crop_oil_palm"])
    elif ctx["sdpt_tree_crop"]:
        set_tokens(tokens, node_codes, crop_reclass_idx, "C", node_code_map["crop_sdpt_tree_crop"])
    elif ctx["pre_2000_plantation"]:
        set_tokens(tokens, node_codes, crop_reclass_idx, "C", node_code_map["crop_pre_2000_plantation"])

    # If no crop extent applies, forest extents are applied by this order: GMW mangrove -> SDPT planted forest
    elif ctx["gmw_mangrove"]:
        set_tokens(tokens, node_codes, forest_reclass_idx, "F", node_code_map["forest_gmw_mangrove"])
    elif ctx["sdpt_planted_forest"]:
        set_tokens(tokens, node_codes, forest_reclass_idx, "F", node_code_map["forest_sdpt_planted_forest"])

In [ ]:
# Tall vegetation all years
def apply_all_tall_veg(ctx):
    tcl_prior = ctx["tcl_prior"]
    driver = ctx["driver"]

    all_idx = range(len(ctx["tokens"]))

    # If TCL has occurred by the start of timeseries and the driver is permanent ag, assume tall veg is tree crops
    if tcl_prior and driver == 1:
        set_tokens(ctx["tokens"], ctx["node_codes"], all_idx, "C", node_code_map["crop_perm_ag_driver"])
    else:
        don't set anything

In [274]:
# Short vegetation all years
def apply_all_short_veg(ctx):
    tcl_prior = ctx["tcl_prior"]
    driver = ctx["driver"]

    all_idx = range(len(ctx["tokens"]))

    # If TCL has occurred by the start of the timeseries and the driver is permanent ag and not in cultivated grass extent, assume crop
    if (tcl_prior and driver == 1 and not ctx["gpw_cultiv_grass"]):
        set_tokens(ctx["tokens"], ctx["node_codes"], all_idx, "C", node_code_map["crop_perm_ag_driver"])

    # If TCL has occurred by the start of the timeseries and the driver is shifting cultivation, logging, wildfire, or other natural disturbances, assume unstocked forest
    elif tcl_prior and driver in (3, 4, 5, 7):
        set_tokens(ctx["tokens"], ctx["node_codes"], all_idx, "F", node_code_map[driver])

    else:
       don't set anything'


In [ ]:
def apply_regex_reclassification_rules(ctx):
    apply_extent_rules(ctx)

    token_seq = "".join(ctx["tokens"])
    if re.fullmatch(r"F+", token_seq):
        apply_all_tall_veg(ctx)
    elif re.fullmatch(r"G+", token_seq):
            apply_all_short_veg
# TODO: Apply extent rules first, if match don't go onto other rules. If not, evaluate the token pattern before deciding which set of rules to apply

In [ ]:
# # Function to apply default rules to go from GLAD land cover to IPCC land Use
# def lc_to_lu_default(lc_vals, rules_table):
#     lu_vals = []
#     for lc in lc_vals:
#         try:
#             lu = lc_token_map[lc]
#             assigned = rules_table["kind"]["default"][lu]["label"]
#             lu_vals.append(assigned)
#         except KeyError:
#             lu_vals.append(None)
#
#     return lu_vals

# Labels for final LU tokens
token_lu_label_map = {
    "F": "Forest land",
    "G": "Grassland",
    "C": "Cropland",
    "S": "Settlements",
    "W": "Wetlands",
    "B": "Other land",
    "O": "Other land",
    "I": "Other land",
    "-": None,
}
# Need to reassign

# Function to apply default rules to go from GLAD land cover to LU tokens
def lc_to_lu_default(lc_vals, rules_table=None):
    return [token_lu_label_map.get(token_for_lc(lc)) for lc in lc_vals]

#TODO: Assign default node codes here

In [275]:
# Iterate through each scenario and classify LC to create LU and node_code timeseries
def classify_dataframe(df, rules, lc_cols):
    out = []

    for idx, row in df.iterrows():
        driver = as_int_or_none(row.get("driver"))
        lc_ts = [row.get(c) for c in lc_cols]

        lu_ts, lu_token_ts, node_code_ts = classify_scenario(
            row.get("id", idx),
            lc_ts,
            rules,
            driver=driver,
            tcl_year=row.get("tcl_year"),
            gpw_cultiv_grass=row.get("gpw_cultiv_grass"),
            pre_2000_plantation=row.get("pre_2000_plantation", row.get("pre_200_plantation")),
            oil_palm=row.get("oil_palm"),
            sdpt_tree_crop=row.get("sdpt_tree_crop"),
            sdpt_planted_forest=row.get("sdpt_planted_forest"),
            gmw_mangrove=row.get("gmw_mangrove"),
        )

        # Create annual land use, token, and node code columns
        lu_cols = {f"lu_{y}": lu_ts[i] for i, y in enumerate(years)}
        lu_token_cols = {f"lu_token_{y}": lu_token_ts[i] for i, y in enumerate(years)}
        node_code_cols = {f"node_code_{y}": node_code_ts[i] for i, y in enumerate(years)}

        # Create time step transitions and conversion flags
        conversion = False
        trans_cols = {}
        for i, (a, b) in enumerate(zip(years[:-1], years[1:])):
            a_lu, b_lu = lu_ts[i], lu_ts[i + 1]
            key = f"{a}_{b}"
            if a_lu == b_lu:
                trans_cols[key] = f"{a_lu} remaining {b_lu}"
            else:
                trans_cols[key] = f"{a_lu} to {b_lu}"
                conversion = True

        out.append({
            "id": row.get("id", idx),
            "driver": driver_code_map.get(driver, str(driver) if driver is not None else None),
            "tcl_year": as_int_or_none(row.get("tcl_year")),
            "gpw_cultiv_grass": as_bool(row.get("gpw_cultiv_grass")),
            "pre_2000_plantation": as_bool(row.get("pre_2000_plantation", row.get("pre_200_plantation"))),
            "oil_palm": as_bool(row.get("oil_palm")),
            "sdpt_tree_crop": as_bool(row.get("sdpt_tree_crop")),
            "sdpt_planted_forest": as_bool(row.get("sdpt_planted_forest")),
            "gmw_mangrove": as_bool(row.get("gmw_mangrove")),
            **lu_cols,
            **lu_token_cols,
            **node_code_cols,
            **trans_cols,
            "conversion_occurred": conversion,
        })

    return pd.DataFrame(out)

In [276]:
# Main function to apply regex rules to the LC timeseries and return:
# - final LU labels
# - final LU tokens
# - node_code timeseries
def classify_scenario(
    scenario_id,
    lc_timeseries,
    rules=None,
    driver=None,
    tcl_year=None,
    gpw_cultiv_grass=False,
    pre_2000_plantation=False,
    oil_palm=False,
    sdpt_tree_crop=False,
    sdpt_planted_forest=False,
    gmw_mangrove=False,
):
    # Create default token array from LC timeseries
    lc_vals = [int(v) for v in lc_timeseries]
    tokens = [token_for_lc(v) for v in lc_vals]
    node_codes = [default_node_code(token) for token in tokens]

    # Inputs for regex and extent rules
    ctx = {
        "scenario_id": scenario_id,
        "lc_vals": lc_vals,
        "tokens": tokens,
        "node_codes": node_codes,
        "driver": as_int_or_none(driver),
        "tcl_year": as_int_or_none(tcl_year),
        "tcl_prior": tcl_prior_to_timeseries(tcl_year, start_year=min(years)),
        "gpw_cultiv_grass": as_bool(gpw_cultiv_grass),
        "pre_2000_plantation": as_bool(pre_2000_plantation),
        "oil_palm": as_bool(oil_palm),
        "sdpt_tree_crop": as_bool(sdpt_tree_crop),
        "sdpt_planted_forest": as_bool(sdpt_planted_forest),
        "gmw_mangrove": as_bool(gmw_mangrove),
    }

    apply_regex_reclassification_rules(ctx)

    final_tokens = ctx["tokens"]
    lu_vals = [token_lu_label_map.get(token) for token in final_tokens]

    return lu_vals, final_tokens, ctx["node_codes"]

In [277]:
# Configuration
file_path = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/conversion_LUC_scenarios.xlsx"
sheet_name = "scenarios"

lc_cols = [f"lc_{y}" for y in years]

In [278]:
# Load data
scenarios_df = pd.read_excel(file_path, sheet_name=sheet_name)

In [279]:
# Run classification
# Coerce scenarios_df to numeric
numeric_cols = lc_cols + ["driver", "tcl_year"]
 for c in numeric_cols:
    if c in scenarios_df.columns:
        scenarios_df[c] = pd.to_numeric(scenarios_df[c], errors="coerce")
    
# Run classification
results_df = classify_dataframe(scenarios_df, rules=None, lc_cols=lc_cols)

# Output results 
print("\nClassification Results:")
print(results_df)


Classification Results:
      id                driver       sdpt_type sdpt_name logging_concession  \
0    1.0                  None            None      None                yes   
1    2.0                  None  planted forest      None               None   
2    3.0                  None            None      None               None   
3    4.0                  None            None      None               None   
4    5.0                  None            None      None               None   
5    6.0               Logging            None      None               None   
6    7.0               Logging            None      None               None   
7    8.0                  None            None      None               None   
8    9.0                  None            None      None               None   
9   10.0                  None            None      None               None   
10  11.0                  None            None      None               None   
11  12.0                  N

In [280]:
results_df.to_excel("/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/conversion_LUC_scenario_results.xlsx", index=False)